Regressor와 Classifier의 차이

Regressor = 연속된 숫자 중 숫자 하나만 출력-> mae, mse, r2 스코어(실제 값을 얼마나 잘 설명했냐) 출력 가능

Classifier =  연속된 구간 중 하나의 구간만 출력-> f-1 스코어(실제 구간에서 얼마나 잘 분류했냐) 출력 가능

이용객수는 다양한 구간으로 나누어 실험해본 결과. test1.csv 파일의 "0~700/700~3000/3000 이상"이 가장 f-1스코어 높게 측정

기존 오월드 csv 파일

In [1]:
# 특이사항 1:   중간 구간(700~3천) 구간이 f1스코어가 다른 구간보다 낮게 출력됨.
#              표본 개수 문제는 아니고 아마 피처 작용하는 것 중 수치로는 알 수 없는 무언가가 있는 것으로 예상(습도, 휴관, 샌드위치 등등등)
#
# 특이사항 2:   구간별로 큰 차이가 없어서 class_weight = "balanced"는 굳이 안해도 될 듯 함. 
#               캣부스트의 경우 오히려 전체 f-1 스코어는 오르지만 중간 구간은 조금 떨어지는 상황 발생
    
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report

# 아래는 쓰지 않을 모델들
#from sklearn.preprocessing import StandardScaler
#from sklearn.linear_model import LogisticRegression
#from sklearn.neighbors import KNeighborsClassifier
#from sklearn.compose import ColumnTransformer
#from sklearn.preprocessing import OneHotEncoder, StandardScaler
#from sklearn.pipeline import Pipeline
df = pd.read_csv("Oworld.csv", encoding="utf-8-sig")


X = df[["holiday","mon","tue","wed","thur","fri","sat","sun","temperature","rain","humidity"]]

#========================================
#이용객수 구간 범위 지정
#0-700명: 여유
#700~3000명 : 혼잡
#3000 이상 : 매우 혼잡
bins = [0, 700, 3000, np.inf]
labels = [0, 1, 2]

y = pd.cut(
    df["visitors"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
y = y.astype(int)

#========================================
# 요일별 순서로 되어있기 때문에 랜덤으로 분리
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


#========================================
# 랜덤포레스트, 캣부스트
models = {
    #선형회귀, KNN 주석 풀면 적용 가능
    #"Logistic Regression": LogisticRegression(max_iter=1000),
    #"KNN": KNeighborsClassifier(n_neighbors=5),

    #"Random Forest": RandomForestClassifier(
    #    #class_weight="balanced",
    #    n_estimators=300,
    #    random_state=42,
    #    n_jobs=-1
    #),
    "CatBoost": CatBoostClassifier(
        #auto_class_weights="Balanced",            
        iterations=500,
        depth=10,
        learning_rate=0.01,
        random_state=42,
        verbose=0
    )
}
#========================================
#모델 학습 후 평가
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)

    f1 = f1_score(y_test,y_pred,average="weighted")
    f1_macro = f1_score(y_test,y_pred,average="macro")
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~3000명",
                "3001명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

#model.save_model("Oworld_catboost_1차_전처리.cbm")


              precision    recall  f1-score   support

      0~700명       0.73      0.87      0.79       102
   700~3000명       0.64      0.64      0.64        85
    3001명 이상       0.97      0.63      0.77        57

    accuracy                           0.73       244
   macro avg       0.78      0.71      0.73       244
weighted avg       0.75      0.73      0.73       244

      Model   f1
0  CatBoost 0.73


테스트 1

In [ ]:
# 테스트 1
    
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report


df = pd.read_csv("test1.csv", encoding="utf-8-sig")
X = df[["dayoff", "nextdayoff", "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec", "temperature", "rain","humidity"]]

#========================================
#이용객수 구간 범위 지정
#0-700명: 여유
#700~3000명 : 혼잡
#3000 이상 : 매우 혼잡
bins = [0,700, 3000, np.inf]
labels = [0, 1, 2]

y = pd.cut(
    df["visitors"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
y = y.astype(int)

#========================================
# 요일별 순서로 되어있기 때문에 랜덤으로 분리
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


#========================================
# 캣부스트
models = {
   "CatBoost": CatBoostClassifier(
        iterations=1164,
        depth=8,
        learning_rate=0.04585878172511418,
        l2_leaf_reg=8.928840273173956,
        bagging_temperature=0.9065380504018735,
        random_strength=0.051733258645387664,
        border_count=49,
        random_state=42,
        verbose = 0
       )
}

#========================================
#모델 학습 후 평가
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)

    f1 = f1_score(y_test, y_pred, average="weighted")       
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~3000명",
                "3001명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

#model.save_model("Oworld_catboost_test1.cbm")

              precision    recall  f1-score   support

      0~700명       0.80      0.82      0.81       102
   700~3000명       0.70      0.74      0.72        85
    3001명 이상       0.94      0.81      0.87        57

    accuracy                           0.79       244
   macro avg       0.81      0.79      0.80       244
weighted avg       0.80      0.79      0.79       244

      Model   f1
0  CatBoost 0.79


In [70]:
#테스트 1 임의의 데이터 성능 테스트
import pandas as pd
from catboost import CatBoostClassifier

#모델 불러오기
model = CatBoostClassifier()
model.load_model("Oworld_catboost_test1.cbm")

test_data = pd.DataFrame([
    {
        "dayoff": 1,
        "nextdayoff": 1,
        "jan": 0,
        "feb": 0,
        "mar": 0,
        "apr": 0,
        "may": 0,
        "jun": 0,
        "jul": 0,
        "aug": 1,
        "sep": 0,
        "oct": 0,
        "nov": 0,
        "dec": 1,
        "temperature": 60,
        "rain": 1,
        "humidity": 100
    }
])


prediction = model.predict(test_data)
prediction = int(prediction[0][0])
print("예측 클래스:", prediction)
if prediction == 0:
    print("예측 결과: 0~700명")
elif prediction == 1:
    print("예측 결과: 700~3000명")
else:
    print("예측 결과: 3001명 이상")

예측 클래스: 1
예측 결과: 700~3000명


테스트1_cat

In [ ]:
# 테스트 1_cat 범주형 데이터이기에 랜덤포레스트 적용 x 
    
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report


df = pd.read_csv("test1_cat.csv", encoding="utf-8-sig")


X = df[["dayoff", "nextdayoff","month", "temperature", "rain","humidity"]]
cat_features = ["month"]
#========================================
#이용객수 구간 범위 지정
#0-700명: 여유
#700~3000명 : 혼잡
#3000 이상 : 매우 혼잡
bins = [0, 700, 3000, np.inf]
labels = [0, 1, 2]

y = pd.cut(
    df["visitors"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
y = y.astype(int)

#========================================
# 요일별 순서로 되어있기 때문에 랜덤으로 분리
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


#========================================
# 캣부스트
models = {

    "CatBoost": CatBoostClassifier(
        #auto_class_weights="Balanced",            
        iterations=500,
        depth=10,
        learning_rate=0.01,
        random_state=42,
        verbose=0
    )
}

#========================================
#모델 학습 후 평가
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train,cat_features=cat_features)
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)
    f1 = f1_score(y_test,y_pred,average="weighted")
    f1_macro = f1_score(y_test,y_pred,average="macro")
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~3000명",
                "3001명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

              precision    recall  f1-score   support

      0~700명       0.78      0.81      0.79       102
   700~3000명       0.66      0.67      0.66        85
    3001명 이상       0.90      0.79      0.84        57

    accuracy                           0.76       244
   macro avg       0.78      0.76      0.77       244
weighted avg       0.76      0.76      0.76       244

      Model   f1
0  CatBoost 0.76


테스트 2

In [ ]:
# 테스트 2
    
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report


df = pd.read_csv("test2.csv", encoding="utf-8-sig")

X = df[["mon", "tue", "wed", "thur", "fri", "sat", "sun", "holiday", "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec", "temperature", "rain","humidity"]]


#========================================
#이용객수 구간 범위 지정
#0-700명: 여유
#700~3000명 : 혼잡
#3000 이상 : 매우 혼잡
bins = [0, 700, 3000, np.inf]
labels = [0, 1, 2]

y = pd.cut(
    df["visitors"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
y = y.astype(int)

#========================================
# 요일별 순서로 되어있기 때문에 랜덤으로 분리
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


#========================================
# 캣부스트
models = {
    "CatBoost": CatBoostClassifier(
        #auto_class_weights="Balanced",            
        iterations=500,
        depth=10,
        learning_rate=0.01,
        random_state=42,
        verbose=0
    )
}

#========================================
#모델 학습 후 평가
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)
    f1 = f1_score(y_test,y_pred,average="weighted")
    f1_macro = f1_score(y_test,y_pred,average="macro")
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~3000명",
                "3001명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

              precision    recall  f1-score   support

      0~700명       0.80      0.86      0.83       102
   700~3000명       0.66      0.74      0.70        85
    3001명 이상       0.95      0.65      0.77        57

    accuracy                           0.77       244
   macro avg       0.80      0.75      0.77       244
weighted avg       0.79      0.77      0.77       244

      Model   f1
0  CatBoost 0.77


테스트2_cat

In [ ]:
# 테스트 2_cat 
    
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report


df = pd.read_csv("test2_cat.csv", encoding="utf-8-sig")


X = df[["weekday","holiday", "month", "temperature", "rain", "humidity"]]
cat_features = ["weekday", "month"]
#========================================
#이용객수 구간 범위 지정
#0-700명: 여유
#700~3000명 : 혼잡
#3000 이상 : 매우 혼잡
bins = [0, 700, 3000, np.inf]
labels = [0, 1, 2]

y = pd.cut(
    df["visitors"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
y = y.astype(int)

#========================================
# 요일별 순서로 되어있기 때문에 랜덤으로 분리
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


#========================================
# 캣부스트
models = {
    "CatBoost": CatBoostClassifier(
        #auto_class_weights="Balanced",            
        iterations=500,
        depth=10,
        learning_rate=0.01,
        random_state=42,
        verbose=0
    )
}

#========================================
#모델 학습 후 평가
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train,cat_features=cat_features)
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)
    f1 = f1_score(y_test,y_pred,average="weighted")
    f1_macro = f1_score(y_test,y_pred,average="macro")
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~3000명",
                "3001명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

              precision    recall  f1-score   support

      0~700명       0.78      0.86      0.82       102
   700~3000명       0.70      0.68      0.69        85
    3001명 이상       0.94      0.79      0.86        57

    accuracy                           0.78       244
   macro avg       0.81      0.78      0.79       244
weighted avg       0.79      0.78      0.78       244

      Model   f1
0  CatBoost 0.78


테스트 3

In [ ]:
# 테스트 3
    
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report


df = pd.read_csv("test3.csv", encoding="utf-8-sig")

X = df[["dayoff", "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec", "temperature", "rain","humidity"]]


#========================================
#이용객수 구간 범위 지정
#0-700명: 여유
#700~3000명 : 혼잡
#3000 이상 : 매우 혼잡
bins = [0, 700, 3000, np.inf]
labels = [0, 1, 2]

y = pd.cut(
    df["visitors"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
y = y.astype(int)

#========================================
# 요일별 순서로 되어있기 때문에 랜덤으로 분리
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


#========================================
# 캣부스트
models = {
    "CatBoost": CatBoostClassifier(
        #auto_class_weights="Balanced",            
        iterations=500,
        depth=10,
        learning_rate=0.01,
        random_state=42,
        verbose=0
    )
}

#========================================
#모델 학습 후 평가
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)
    f1 = f1_score(y_test,y_pred,average="weighted")
    f1_macro = f1_score(y_test,y_pred,average="macro")
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~3000명",
                "3001명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

              precision    recall  f1-score   support

      0~700명       0.81      0.83      0.82       102
   700~3000명       0.65      0.74      0.69        85
    3001명 이상       0.95      0.70      0.81        57

    accuracy                           0.77       244
   macro avg       0.80      0.76      0.77       244
weighted avg       0.79      0.77      0.77       244

      Model   f1
0  CatBoost 0.77


테스트3_cat

In [ ]:
# 테스트 3_cat
    
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report


df = pd.read_csv("test3_cat.csv", encoding="utf-8-sig")


X = df[["dayoff", "month", "temperature", "rain", "humidity"]]
cat_features = ["month"]
#========================================
#이용객수 구간 범위 지정
#0-700명: 여유
#700~3000명 : 혼잡
#3000 이상 : 매우 혼잡
bins = [0, 700, 3000, np.inf]
labels = [0, 1, 2]

y = pd.cut(
    df["visitors"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
y = y.astype(int)

#========================================
# 요일별 순서로 되어있기 때문에 랜덤으로 분리
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


#========================================
# 캣부스트
models = {
    "CatBoost": CatBoostClassifier(
        #auto_class_weights="Balanced",            
        iterations=500,
        depth=10,
        learning_rate=0.01,
        random_state=42,
        verbose=0
    )
}
#========================================
#모델 학습 후 평가
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train,cat_features=cat_features)
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)
    f1 = f1_score(y_test,y_pred,average="weighted")
    f1_macro = f1_score(y_test,y_pred,average="macro")
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~3000명",
                "3001명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

              precision    recall  f1-score   support

      0~700명       0.80      0.80      0.80       102
   700~3000명       0.65      0.72      0.68        85
    3001명 이상       0.90      0.75      0.82        57

    accuracy                           0.76       244
   macro avg       0.78      0.76      0.77       244
weighted avg       0.77      0.76      0.76       244

      Model   f1
0  CatBoost 0.76


테스트4

In [ ]:
# 테스트 4
    
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report


df = pd.read_csv("test4.csv", encoding="utf-8-sig")

X = df[["weekend", "holiday", "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec", "temperature", "rain","humidity"]]


#========================================
#이용객수 구간 범위 지정
#0-700명: 여유
#700~3000명 : 혼잡
#3000 이상 : 매우 혼잡
bins = [0, 700, 3000, np.inf]
labels = [0, 1, 2]

y = pd.cut(
    df["visitors"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
y = y.astype(int)

#========================================
# 요일별 순서로 되어있기 때문에 랜덤으로 분리
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


#========================================
# 캣부스트
models = {
    "CatBoost": CatBoostClassifier(
        #auto_class_weights="Balanced",            
        iterations=500,
        depth=10,
        learning_rate=0.01,
        random_state=42,
        verbose=0
    )
}

#========================================
#모델 학습 후 평가
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)
    f1 = f1_score(y_test,y_pred,average="weighted")
    f1_macro = f1_score(y_test,y_pred,average="macro")
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~3000명",
                "3001명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

              precision    recall  f1-score   support

      0~700명       0.79      0.87      0.83       102
   700~3000명       0.67      0.73      0.70        85
    3001명 이상       0.95      0.67      0.78        57

    accuracy                           0.77       244
   macro avg       0.81      0.76      0.77       244
weighted avg       0.79      0.77      0.77       244

      Model   f1
0  CatBoost 0.77


테스트4_cat

In [ ]:
# 테스트 4_cat
    
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report


df = pd.read_csv("test4_cat.csv", encoding="utf-8-sig")


X = df[["weekend", "holiday", "month", "temperature", "rain", "humidity"]]
cat_features = ["month"]
#========================================
#이용객수 구간 범위 지정
#0-700명: 여유
#700~3000명 : 혼잡
#3000 이상 : 매우 혼잡
bins = [0, 700, 3000, np.inf]
labels = [0, 1, 2]

y = pd.cut(
    df["visitors"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
y = y.astype(int)

#========================================
# 요일별 순서로 되어있기 때문에 랜덤으로 분리
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


#========================================
# 캣부스트
models = {
    "CatBoost": CatBoostClassifier(
        #auto_class_weights="Balanced",            
        iterations=500,
        depth=10,
        learning_rate=0.01,
        random_state=42,
        verbose=0
    )
}

#========================================
#모델 학습 후 평가
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train,cat_features=cat_features)
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)
    f1 = f1_score(y_test,y_pred,average="weighted")
    f1_macro = f1_score(y_test,y_pred,average="macro")
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~3000명",
                "3001명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

              precision    recall  f1-score   support

      0~700명       0.79      0.80      0.80       102
   700~3000명       0.63      0.73      0.68        85
    3001명 이상       0.95      0.70      0.81        57

    accuracy                           0.75       244
   macro avg       0.79      0.75      0.76       244
weighted avg       0.77      0.75      0.76       244

      Model   f1
0  CatBoost 0.76
